In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

In [2]:
import pandas as pd

fname = 'eastman_dye_data_rgb_hex.csv'
df = pd.read_csv(fname)
df

,Compound_name,Outcome,SMILES,RGB,hex,HSV
0,12832,698.0,COc1ccc2cc(C3=[N+]4C(=Nc5c(-c6ccc7ccccc7c6)c(B...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)"
1,5722,664.0,COCCOCCOCCOc1ccc(/C=C/C2=[N+]3C(=C(c4cc(OC)cc(...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)"
2,19513,656.0,COc1c2ccccc2c(OC)c2c(C#Cc3ccccc3)c3cc4sccc4cc3...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)"
3,19121,683.0,CC(C)[Si](C#Cc1c2cc3ccccc3cc2c(C#C[Si](C(C)C)(...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)"
4,3280; 17834; 20267; 20501,676.0,CCOC(=O)COc1ccc(/C=C/C2=C3C(C)=CC(/C=C/c4ccc(O...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)"
...,...,...,...,...,...,...
1545,8151,380.0,CCn1c2ccc(-c3ccc(-c4cccs4)s3)cc2c2cc(-c3ccc(-c...,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)"
1546,15056,380.0,C[Si]1(C)O[Si](C)(C)c2cc3c(-c4ccccc4F)c4cc(Cl)...,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)"
1547,17263,380.0,O=S1(=O)c2ccccc2C2(c3cc(-c4ccc(N(c5ccccc5)c5cc...,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)"
1548,1199,380.0,COc1ccc(/C=C/c2nc(-c3ccccc3)[nH]c2/C=C/c2ccc(O...,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)"


### 1) Clean/salsa-fy the SMILES.

In [3]:
def undo_BrCl_singles(smi):
    smi = smi.replace('R','Br')
    return smi.replace('L','Cl')
def do_BrCl_singles(smi):
    smi = smi.replace('Br','R')
    return smi.replace('Cl','L')   

tokens_salsa = '#%()+-0123456789<=>BCFHILNOPRSX[]cnos'
tokens_dyes = 'Ub)gsP5u=8H[d]noW06%2GLlO+NFSTa9ZCeI-r4(1ihAVBRc3M7pt#'

In [17]:
import os
from tqdm.notebook import tqdm
from utilities.rdkit_utils import get_cansmiles

ddir = 'data/model_ready/dyes/'

cuts = ['full'] #,'val','test']

all_smis = []
raw_smis = []

for i,cut in enumerate(cuts):
    
    os.makedirs(f'{ddir}/{cut}/', exist_ok=True)
    
    smis = []
    labs = []
    
    keep_idc = []
    bad_idc = []
    
    for idx,tup in tqdm(enumerate(df.itertuples()), total=len(df)):        
        smi = tup.SMILES
        target = tup.Outcome
        
        _smi = get_cansmiles(smi, iso=False)
        components = _smi.split('.')
        components = sorted(components, key=lambda x: len(x))
        
        if len(components) > 1:
            bad_idc.append(idx)
            continue
            
        smi = components[-1]
        
        if smi:
            raw_smis.append(smi)
            
            # Need to match SALSA's vocab !!!
            smi = do_BrCl_singles(smi)  
            if not all(c in tokens_salsa for c in smi):
                bad_idc.append(idx)
                    
            elif len(smi) > 120:
                bad_idc.append(idx)
            else:
                keep_idc.append(idx)
                
assert(set(keep_idc).isdisjoint(bad_idc))

  0%|          | 0/1550 [00:00<?, ?it/s]

In [13]:
df_clean = df.loc[keep_idc]
df_clean['cansmi'] = df_clean.SMILES.apply(lambda x: get_cansmiles(x))
df_clean = df_clean.reset_index(drop=True)
df_clean

,Compound_name,Outcome,SMILES,RGB,hex,HSV,cansmi
0,12832,698.0,COc1ccc2cc(C3=[N+]4C(=Nc5c(-c6ccc7ccccc7c6)c(B...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)",COc1ccc2cc(C3=[N+]4C(=Nc5c(-c6ccc7ccccc7c6)c(B...
1,5722,664.0,COCCOCCOCCOc1ccc(/C=C/C2=[N+]3C(=C(c4cc(OC)cc(...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)",COCCOCCOCCOc1ccc(C=CC2=[N+]3C(=C(c4cc(OC)cc(OC...
2,19513,656.0,COc1c2ccccc2c(OC)c2c(C#Cc3ccccc3)c3cc4sccc4cc3...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)",COc1c2ccccc2c(OC)c2c(C#Cc3ccccc3)c3cc4sccc4cc3...
3,3280; 17834; 20267; 20501,676.0,CCOC(=O)COc1ccc(/C=C/C2=C3C(C)=CC(/C=C/c4ccc(O...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)",CCOC(=O)COc1ccc(C=CC2=C3C(C)=CC(C=Cc4ccc(OCC(=...
4,20413,654.0,Cc1ccc(C2=C3C=C(c4ccc(C)s4)C(c4ccc(C)s4)=[N+]3...,"[1.0, 0.0, 0.0]",#ff0000,"(0.0, 1.0, 1.0)",Cc1ccc(C2=C3C=C(c4ccc(C)s4)C(c4ccc(C)s4)=[N+]3...
...,...,...,...,...,...,...,...
1389,1624,380.0,c1ccc(-c2nc(-c3c[nH]c4ccccc34)n3ccccc23)cc1,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)",c1ccc(-c2nc(-c3c[nH]c4ccccc34)n3ccccc23)cc1
1390,8151,380.0,CCn1c2ccc(-c3ccc(-c4cccs4)s3)cc2c2cc(-c3ccc(-c...,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)",CCn1c2ccc(-c3ccc(-c4cccs4)s3)cc2c2cc(-c3ccc(-c...
1391,17263,380.0,O=S1(=O)c2ccccc2C2(c3cc(-c4ccc(N(c5ccccc5)c5cc...,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)",O=S1(=O)c2ccccc2C2(c3cc(-c4ccc(N(c5ccccc5)c5cc...
1392,1199,380.0,COc1ccc(/C=C/c2nc(-c3ccccc3)[nH]c2/C=C/c2ccc(O...,"[0.3, 0.0, 0.3]",#4c004c,"(0.8333333333333334, 1.0, 0.2980392156862745)",COc1ccc(C=Cc2nc(-c3ccccc3)[nH]c2C=Cc2ccc(OC)cc...


In [20]:
pdir = 'data/dye_data/hex_cansmi_salsified.csv'
df_clean.to_csv(pdir,index=False)

In [18]:
pdir = 'data/model_ready/dyes/full/anchor_smiles.csv'

df_save = df_clean[['cansmi']]
df_save.columns = ['smiles']

df_save.to_csv(pdir,index=False)